## Import Libraries

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from PIL import Image
from tqdm import tqdm

## MipNeRF bounding

In [ ]:
def apply_mipnerf360_contraction(x):
    norm = torch.norm(x, dim=-1, keepdim=True)
    return torch.where(norm <= 1.0, x, (2.0 - 1.0 / norm) * (x / norm))

## Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, L_pos=10, L_dir=4):
        super().__init__()
        self.freqs_pos = 2.0 ** torch.linspace(0, L_pos - 1, L_pos)
        self.freqs_dir = 2.0 ** torch.linspace(0, L_dir - 1, L_dir)

    def forward(self, x, freqs):
        out = [x]
        for freq in freqs.to(x.device):
            out.append(torch.sin(x * freq))
            out.append(torch.cos(x * freq))
        return torch.cat(out, dim=-1)

## NeRF model

In [ ]:
class NeRFMLP(nn.Module):
    def __init__(self, L_pos=10, L_dir=4, hidden_dim=256):
        super().__init__()
        self.encoder = PositionalEncoding(L_pos, L_dir)
        in_pos = 3 + (3 * 2 * L_pos)
        in_dir = 3 + (3 * 2 * L_dir)
        
        self.layer1 = nn.Linear(in_pos, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)
        self.layer4 = nn.Linear(hidden_dim, hidden_dim)
        self.layer5 = nn.Linear(hidden_dim + in_pos, hidden_dim)
        self.layer6 = nn.Linear(hidden_dim, hidden_dim)
        self.layer7 = nn.Linear(hidden_dim, hidden_dim)
        self.layer8 = nn.Linear(hidden_dim, hidden_dim)
        
        self.sigma_out = nn.Linear(hidden_dim, 1)
        self.feature_out = nn.Linear(hidden_dim, hidden_dim)
        self.rgb_layer1 = nn.Linear(hidden_dim + in_dir, hidden_dim // 2)
        self.rgb_out = nn.Linear(hidden_dim // 2, 3)

    def forward(self, points, dirs):
        contracted_points = apply_mipnerf360_contraction(points)
        pos_enc = self.encoder(contracted_points, self.encoder.freqs_pos)
        dir_enc = self.encoder(dirs, self.encoder.freqs_dir)
        
        h = F.relu(self.layer1(pos_enc))
        h = F.relu(self.layer2(h))
        h = F.relu(self.layer3(h))
        h = F.relu(self.layer4(h))
        h = F.relu(self.layer5(torch.cat([h, pos_enc], dim=-1)))
        h = F.relu(self.layer6(h))
        h = F.relu(self.layer7(h))
        h = self.layer8(h)
        
        sigma = F.softplus(self.sigma_out(h))
        geo_feature = self.feature_out(h)
        
        h_color = torch.cat([geo_feature, dir_enc], dim=-1)
        h_color = F.relu(self.rgb_layer1(h_color))
        rgb = torch.sigmoid(self.rgb_out(h_color))
        
        return rgb, sigma

## Convert Pixels to Rays

In [ ]:
def get_rays(H, W, focal, pose):
    i, j = torch.meshgrid(torch.linspace(0, W-1, W), torch.linspace(0, H-1, H), indexing='xy')
    dirs = torch.stack([(i - W * 0.5) / focal, -(j - H * 0.5) / focal, -torch.ones_like(i)], -1).to(pose.device)
    rays_d = torch.sum(dirs[..., None, :] * pose[:3, :3], -1)
    rays_o = pose[:3, 3].expand(rays_d.shape)
    return rays_o, rays_d

def render_rays(model, rays_o, rays_d, bounds, num_samples=64):
    near, far = bounds[0], bounds[1]
    t_vals = torch.linspace(0., 1., steps=num_samples).to(rays_o.device)
    z_vals = near + (far - near) * t_vals

    pts = rays_o[..., None, :] + rays_d[..., None, :] * z_vals[..., :, None]
    dirs = rays_d[..., None, :].expand(pts.shape)
    
    pts_flat = pts.reshape(-1, 3)
    dirs_flat = dirs.reshape(-1, 3)
    
    rgb_flat, sigma_flat = model(pts_flat, dirs_flat)
    
    rgb = rgb_flat.reshape(pts.shape[:2] + (3,))
    sigma = sigma_flat.reshape(pts.shape[:2])
    
    dists = z_vals[..., 1:] - z_vals[..., :-1]
    dists = torch.cat([dists, torch.tensor([1e10]).expand(dists[..., :1].shape).to(rays_o.device)], -1)
    
    alpha = 1.0 - torch.exp(-sigma * dists)
    transmittance = torch.cumprod(torch.cat([torch.ones((alpha.shape[0], 1)).to(rays_o.device), 1. - alpha + 1e-10], -1), -1)[:, :-1]
    weights = alpha * transmittance
    
    rgb_map = torch.sum(weights[..., None] * rgb, -2)
    return rgb_map

## Testing Module

In [ ]:
def test_nerf():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running Inference/Testing on: {device}")
    
    test_img_dir = "/kaggle/input/datasets/thnhdg/testing/360_v2/bicycle/images_2"
    poses_path = "/kaggle/input/datasets/thnhdg/testing/360_v2/bicycle/poses_bounds.npy"
    weights_path = "/kaggle/working/best_nerf_weights.pth"
    output_dir = "/kaggle/working/render_output"
    os.makedirs(output_dir, exist_ok=True)
 
    downsample_factor = 4 
    raw_poses_bounds = np.load(poses_path)
    poses = raw_poses_bounds[:, :-2].reshape([-1, 3, 5])
    bounds_arr = raw_poses_bounds[:, -2:]
    
    H = int(poses[0, 0, 4] // downsample_factor)
    W = int(poses[0, 1, 4] // downsample_factor)
    focal = poses[0, 2, 4] / downsample_factor
    
    model = NeRFMLP().to(device)
    if os.path.exists(weights_path):
        model.load_state_dict(torch.load(weights_path, map_location=device))
        print("Successfully loaded trained weights.")
    else:
        print(f"Warning: Checkpoint file '{weights_path}' not found. Testing with initialized weights.")
    
    model.eval()
    
    test_files = sorted(os.listdir(test_img_dir))[:15]
    total_test_psnr = 0.0
    CHUNK_SIZE = 4096 
    
    print(f"Starting rendering for {len(test_files)} test views...")
    
    with torch.no_grad():
        for idx, img_name in enumerate(tqdm(test_files, desc="Rendering Test Images")):
            
            gt_path = os.path.join(test_img_dir, img_name)
            gt_img = Image.open(gt_path).convert("RGB").resize((W, H), Image.Resampling.LANCZOS)
            gt_tensor = torch.from_numpy(np.array(gt_img)).float().to(device) / 255.0
            
            pose = torch.from_numpy(poses[idx, :3, :4]).float().to(device)
            bounds = torch.from_numpy(bounds_arr[idx]).float().to(device)
            
            rays_o, rays_d = get_rays(H, W, focal, pose)
            rays_o_flat = rays_o.reshape(-1, 3)
            rays_d_flat = rays_d.reshape(-1, 3)
            
            rendered_pixels = []

            for i in range(0, rays_o_flat.shape[0], CHUNK_SIZE):
                ro_chunk = rays_o_flat[i : i + CHUNK_SIZE]
                rd_chunk = rays_d_flat[i : i + CHUNK_SIZE]
                
                rgb_chunk = render_rays(model, ro_chunk, rd_chunk, bounds)
                rendered_pixels.append(rgb_chunk)

            full_rgb_flat = torch.cat(rendered_pixels, dim=0)
            rendered_img_tensor = full_rgb_flat.reshape(H, W, 3).clamp(0.0, 1.0)
 
            img_mse = F.mse_loss(rendered_img_tensor, gt_tensor)
            img_psnr = -10.0 * torch.log10(img_mse)
            total_test_psnr += img_psnr.item()

            rendered_np = (rendered_img_tensor.cpu().numpy() * 255.0).astype(np.uint8)
            output_img = Image.fromarray(rendered_np)
            output_img.save(os.path.join(output_dir, f"render_{img_name}"))
            
    mean_test_psnr = total_test_psnr / len(test_files)
    print(f"\nEvaluation Complete!")
    print(f"Mean Test Dataset PSNR: {mean_test_psnr:.2f} dB")
    print(f"Rendered outputs saved to folder: {output_dir}")

## Main check

In [ ]:
if __name__ == "__main__":
    test_nerf()